# PTDLNC – Hệ thống ra quyết định giữ chân khách hàng đa kênh (Online Retail II)

Notebook chạy tuần tự **GĐ1 → GĐ8** và **s9 (kiểm định ngoài)**. Mặc định các ô code **ĐỌC kết quả có sẵn** trong `outputs/` (bảng, hình) nên chạy nhanh trên Colab;
phần chạy lại pipeline nằm dưới cờ `RUN_FULL = False`.

**Cảnh báo.** Trạng thái đã đóng băng: SHA-256 của `src/`, `configs/`, `models/`, `CLAUDE.md` và `data/processed/*.parquet` được ghi trong `outputs/manifests/s8_freeze_manifest.json`.
Đặt `RUN_FULL = True` sẽ **ghi đè** các file đó (⇒ hash không còn khớp) và **không thể mở lại Locked Test** (khóa `s8_locked_opened.lock`). Chỉ dùng trên **bản sao**.

**CLO liên quan:** CLO1–CLO5 (đề: CLO1 → phần I–III; CLO2 → I, IV; CLO3 → III, IV; CLO4 → II, IV; CLO5 → IV, V).

In [ ]:
import os, sys, json, hashlib, subprocess, platform
from pathlib import Path

RUN_FULL = False          # True: chạy lại pipeline (GHI ĐÈ data/processed, models, outputs – làm lệch SHA-256 đóng băng). Chỉ dùng trên BẢN SAO.
INSTALL_DEPS = "google.colab" in sys.modules   # tự cài thư viện trên Colab
REPO_URL = "https://github.com/imty-cloud/retention-decision-system.git"  # điền URL GitHub của repo (chỉ cần khi chạy trên Colab và chưa clone)

def find_root():
    p = Path.cwd()
    for q in [p, *p.parents]:
        if (q / "configs" / "config.yaml").exists():
            return q
    return None

ROOT = find_root()
if ROOT is None and REPO_URL:
    subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
    ROOT = Path("/content/repo")
assert ROOT is not None, "Không tìm thấy thư mục dự án (cần configs/config.yaml). Clone repo hoặc đặt REPO_URL."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Thư mục dự án:", ROOT, "| Python", platform.python_version())

## 0. Cài đặt và kiểm SHA-256 so với `s8_freeze_manifest.json`

`requirements_lock.txt` được tạo bằng `pip freeze` trên **Python 3.14.7** (macOS Apple Silicon). Trên Colab (Python thấp hơn) các phiên bản ghim có thể không cài được;
khi đó ô dưới tự rơi về `requirements.txt` (ràng buộc `>=`). Phần chỉ-đọc của notebook chỉ cần `pandas` và `matplotlib`.

In [ ]:
if INSTALL_DEPS:
    lock = ROOT / "outputs" / "manifests" / "requirements_lock.txt"
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(lock)])
    if r.returncode != 0:
        print("Không cài được bản ghim (khác phiên bản Python) → dùng requirements.txt")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")])
else:
    print("Bỏ qua cài đặt (không phải Colab). Tự cài: pip install -r outputs/manifests/requirements_lock.txt")

def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

man = json.loads((ROOT / "outputs/manifests/s8_freeze_manifest.json").read_text(encoding="utf-8"))
def check(entries, kind):
    bad, miss = [], []
    for e in entries:
        p = ROOT / e["file"]
        if not p.exists():
            miss.append(e["file"])
        elif sha256_file(p) != e["sha256"]:
            bad.append(e["file"])
    agg = hashlib.sha256("\n".join(f"{e['file']}:{e['sha256']}" for e in entries).encode()).hexdigest()
    print(f"{kind}: {len(entries)} file | lệch {len(bad)} | thiếu {len(miss)} | hash tổng hợp manifest {agg[:16]}…")
    return bad, miss
b1, m1 = check(man["files"], "mã/mô hình/config/CLAUDE.md")
b2, m2 = check(man["data_files"], "dữ liệu data/processed")
print("ĐÓNG BĂNG KHỚP" if not (b1 or m1 or b2 or m2) else f"CÓ LỆCH: {b1 + b2} | THIẾU: {m1 + m2}")
print("Python khi đóng băng:", man["environment"]["python"], "| hiện tại:", platform.python_version())

In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
try:
    from IPython.display import display
except ImportError:          # chạy ngoài Jupyter
    display = print
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 70)
TABLES, FIGS = ROOT / "outputs" / "tables", ROOT / "outputs" / "figures"

def show_table(name, n=12):
    df = pd.read_csv(TABLES / name)
    print(f"[{name}] {df.shape[0]} dòng × {df.shape[1]} cột")
    display(df.head(n).round(4))

def show_fig(name, width=11):
    img = mpimg.imread(FIGS / name)
    h = width * img.shape[0] / img.shape[1]
    plt.figure(figsize=(width, h)); plt.imshow(img); plt.axis("off"); plt.title(name, fontsize=9); plt.show()

def run(cmd):
    print("$", cmd)
    if RUN_FULL:
        subprocess.run(cmd, shell=True, check=True, cwd=ROOT)
    else:
        print("   (bỏ qua – RUN_FULL = False)")

## GĐ1 – Thiết kế bài toán và dữ liệu, chống rò rỉ (s1, s1b)

**Mục tiêu.** Định nghĩa đơn vị khách × t0, chặn dữ liệu cutoff = t0 − 1 ngày, nhãn mua lại 30 ngày, chia tập có embargo và entity holdout; sinh lớp can thiệp bán tổng hợp.

**Phương pháp.** Làm sạch Online Retail II → snapshot point-in-time → data contract (pandera) → 6 kiểm thử rò rỉ → lớp can thiệp (Y(0) thật, Y(a) từ cơ chế đã biết, 20 % nhánh ngẫu nhiên).

**Liên quan:** Chương 1 · mục III · **CLO:** CLO1, CLO2 · **Lệnh tái lập:** `python src/s1_data.py` ; `python src/s1b_intervention.py`

In [ ]:
run("python src/s1_data.py")
run("python src/s1b_intervention.py")
show_table("s1_split_summary.csv", 12)
show_table("s1_leakage_tests.csv", 6)
show_table("s1b_confounding_check.csv", 12)
show_fig("fig_s1_split_timeline.png")
show_fig("fig_s1_base_rate_by_t0.png")

## GĐ2 – Dự báo nền và hiệu chỉnh xác suất (s2)

**Mục tiêu.** μ0(x) = P(mua lại | a0, x) và giá trị giỏ Ŝ có khoảng conformal; hiệu chỉnh để chi phí chiết khấu không bị lệch.

**Phương pháp.** Logistic và LightGBM (GroupKFold theo khách) → Platt/Isotonic khớp trên calib → ROC-AUC, PR-AUC, log loss, Brier, ECE 15 bin; conformal 90 % cho Ŝ.

**Liên quan:** Chương 2 · mục IV.1 · **CLO:** CLO2 · **Lệnh tái lập:** `python src/s2_predict.py`

In [ ]:
run("python src/s2_predict.py")
show_table("s2_metrics_valid.csv", 6)
show_table("s2_acceptance.csv", 12)
show_fig("fig_s2_reliability_valid.png")
show_fig("fig_s2_basket_conformal.png")

## GĐ3 – Giải thích và độ tin cậy (s3)

**Mục tiêu.** Giải thích mô hình μ0 và sinh mã lý do top-3 cho Router; kiểm tra ổn định và faithfulness.

**Phương pháp.** PFI (cá nhân và theo nhóm), SHAP TreeExplainer (thang log-odds), ổn định hạng SHAP qua bootstrap, thử xóa top-k. SHAP/PFI không phải nhân quả.

**Liên quan:** Chương 3 · mục IV.2 · **CLO:** CLO2 · **Lệnh tái lập:** `python src/s3_explain.py`

In [ ]:
run("python src/s3_explain.py")
show_table("s3_pfi_group.csv", 12)
show_table("s3_faithfulness_deletion.csv", 12)
show_fig("fig_s3_pfi.png")
show_fig("fig_s3_shap_beeswarm.png")

## GĐ4 – Nhân quả: ATE, CATE, uplift (s4, s4b)

**Mục tiêu.** Ước lượng tác động của từng voucher so với không làm gì, ở mức trung bình (ATE) và theo khách (CATE); dự báo power cho cổng G3.

**Phương pháp.** Propensity mô hình hóa vs ghi log; ATE naive/IPW/DML; T/X/DR-learner và CausalForestDML; Qini trên nhánh ngẫu nhiên; placebo. s4b: power của G3 trên locked (đăng ký trước).

**Liên quan:** Chương 4 · mục IV.3, V.3 · **CLO:** CLO3 · **Lệnh tái lập:** `python src/s4_causal.py` ; `python src/s4b_power_g3.py`

In [ ]:
run("python src/s4_causal.py")
run("python src/s4b_power_g3.py")
show_table("s4_acceptance.csv", 12)
show_table("s4_qini.csv", 8)
show_table("s4_power_g3_summary.csv", 12)
show_fig("fig_s4_ate_forest.png")
show_fig("fig_s4_qini_curves.png")
show_fig("fig_s4b_power_g3.png")

## GĐ5 – Predict-Then-Optimize và SPO+ (s5, s5b)

**Mục tiêu.** So sánh PTO hai giai đoạn với SPO+ trên regret quyết định; thí nghiệm MSE vs regret; ước lượng DR/AIPW cho giá trị chính sách.

**Phương pháp.** Lô 100 khách, ràng buộc co theo tỷ lệ; vector chi phí DR ngoài fold; 5 seed; quy tắc mở rộng biên lưới đăng ký trước; kiểm chứng cài đặt trên instance 6 khách.

**Liên quan:** Chương 5 · mục IV.4, V.4 · **CLO:** CLO4 · **Lệnh tái lập:** `python src/s5_spo.py` ; `S5_LUOI=mo_rong python src/s5_spo.py` ; `python src/s5b_dr_diagnostics.py`

In [ ]:
run("python src/s5_spo.py")
run("S5_LUOI=mo_rong python src/s5_spo.py")
run("python src/s5b_dr_diagnostics.py")
show_table("s5_regret_summary.csv", 9)
show_table("s5_spo_vs_pto_paired.csv", 12)
show_table("s5_verification_6_summary.csv", 12)
show_fig("fig_s5_regret_by_method.png")
show_fig("fig_s5_mse_vs_regret.png")
show_fig("fig_s5b_dr_z_scores.png")

## GĐ6 – MILP toàn quy mô (s6)

**Mục tiêu.** Giải bài toán phân bổ ngân sách/ký hạn mức gọi/độ phủ cho toàn bộ khách của mỗi t0; đo thời gian và độ nhạy; tác động của hiệu chỉnh lên chi phí chiết khấu.

**Phương pháp.** PuLP + HiGHS (gap 0.1 %), so với Greedy và luật cũ; độ nhạy B, K, m; giá bóng từ LP relaxation; thời gian giải tới n = 50 000.

**Liên quan:** Chương 6 · mục IV.4, V.2 · **CLO:** CLO4 · **Lệnh tái lập:** `python src/s6_milp.py`

In [ ]:
run("python src/s6_milp.py")
show_table("s6_main_summary.csv", 12)
show_table("s6_calibration_discount_cost_summary.csv", 12)
show_table("s6_scaling.csv", 14)
show_fig("fig_s6_main_comparison.png")
show_fig("fig_s6_scaling.png")
show_fig("fig_s6_calibration_discount_cost.png")

## GĐ7 – Action Router, guardrail, HITL (s7)

**Mục tiêu.** Đưa đề xuất MILP qua guardrail cứng, HITL mô phỏng, nhóm đối chứng, bản ghi quyết định có khóa chống lặp. Router v2.1 (đăng ký trước).

**Phương pháp.** Hợp đồng đầu vào → guardrail (opt-out, cancel_rate, frequency cap, sàn biên) → HITL (quy tắc mô phỏng) → đối chứng 10 % → bù độ phủ → sửa trần a2.

**Liên quan:** Chương 7 · mục IV.5 · **CLO:** CLO4, CLO5 · **Lệnh tái lập:** `python src/s7_router.py` ; `python src/s7_router.py --v2` ; `python src/s7_router.py --v21`

In [ ]:
run("python src/s7_router.py")
run("python src/s7_router.py --v2")
run("python src/s7_router.py --v21")
show_table("s7_policy_value_summary_v21.csv", 12)
show_table("s7_decision_flow_v21.csv", 12)
show_table("s7_latency_percentiles_v21.csv", 12)
show_fig("fig_s7_policy_value.png")
show_fig("fig_s7_decision_flow.png")

## GĐ8 – Triển khai, đo lường thực chứng, giám sát MLOps (s8, s8b)

**Mục tiêu.** Đóng băng pipeline, mở Locked Test đúng một lần và chấm Release Gates G1–G7; phát lại OOT (mùa Q4, trôi thật) để đo độ trễ, PSI, Brier/ECE khi nhãn trưởng thành,
Action TVD, giá trị thực hiện bằng DR; so Current với Challenger; kịch bản trôi; khuyết nhãn có chọn lọc; playbook.

**Phương pháp.** Manifest SHA-256 + khóa mở một lần (O_EXCL); `read_rows` chặn locked/oot ngoài phiên được phép; Router v2.1 tuần tự từng t0; Challenger = Platt hiệu chỉnh lại theo kỳ chỉ dùng nhãn đã trưởng thành.

**Liên quan:** Chương 8 · mục IV.6–IV.7, V, VI · **CLO:** CLO5

> **Locked đã mở và OOT đã chạy; khóa `outputs/manifests/s8_locked_opened.lock` đang tồn tại.** Các lệnh `--open-locked`, `--oot` **bị từ chối** nếu chạy lại. Phần dưới chỉ liệt kê lệnh diễn tập/tái lập được.

In [ ]:
run("python src/s8_mlops.py --dry-run")                 # diễn tập: calib đóng vai locked (nhãn THỬ)
run("python src/s8b_oot.py --pilot")                    # thiết kế thí điểm (cỡ mẫu)
run("python src/s8b_oot.py --oot-rehearsal")            # diễn tập OOT: valid = tham chiếu, calib = OOT
print("Khóa locked tồn tại:", (ROOT / "outputs/manifests/s8_locked_opened.lock").exists())
print(open(ROOT / "outputs/manifests/s8_locked_opened.lock", encoding="utf-8").read())

### Release Gates trên Locked Test (mục IV.6)

Bảng đúng là `s8_locked_gate_summary_corrected.csv` (lập từ `s8_locked_gates.csv`); `s8_locked_gate_summary.csv` in toàn `n/a` do lỗi so sánh `is True` với `numpy.bool_`. Kết quả: **G4b trượt ⇒ không đạt đủ cổng**.

In [ ]:
show_table("s8_locked_gate_summary_corrected.csv")
gates = pd.read_csv(TABLES / "s8_locked_gates.csv")
display(gates[gates.quan_the == "toan_bo"][["gate", "dat", "gia_tri", "nguong"]])
show_table("s8_locked_g3_qini.csv", 9)
show_fig("fig_s8_locked_gates.png")

### Phát lại OOT, độ trễ nhãn, Current vs Challenger, trôi, khuyết nhãn, playbook (mục IV.7, V, VI)

In [ ]:
lat = pd.read_csv(TABLES / "s8_oot_latency.csv")
display(lat[lat.ky == "Gộp mọi kỳ"][["khau", "so_luot", "P50", "P95", "P99"]].round(3))
psi = pd.read_csv(TABLES / "s8_oot_psi.csv"); g = psi[psi.ky == "Gộp mọi kỳ"]
print("PSI (gộp):", g.trang_thai.value_counts().to_dict())
display(g.sort_values("psi", ascending=False).head(8)[["loai", "bien", "psi", "trang_thai"]].round(3))
show_table("s8_oot_calibration_matured.csv", 15)
show_table("s8_oot_action_tvd.csv", 5)
show_table("s8_oot_label_delay_by_date.csv", 6)
show_table("s8_oot_label_delay_by_metric.csv", 8)
show_table("s8_oot_dr_value.csv", 6)
show_table("s8_oot_drift_guardrail.csv", 6)
show_table("s8_oot_selective_effect.csv", 3)
show_table("s8_oot_selective_calibration.csv", 6)
show_table("s8_oot_playbook.csv", 9)
show_fig("fig_s8_oot.png")

In [ ]:
show_table("s8_pilot_g3.csv", 6)
show_table("s8_pilot_dr.csv", 10)
show_fig("fig_s8_pilot_sample_size.png")

### Phụ lục SHA-256 và công khai sai lệch

`outputs/tables/s8_sha256_appendix.csv` liệt kê SHA-256 từng file đóng băng, hash tổng hợp, nội dung file khóa và ghi chú sửa sau băng.
`outputs/tables/summary_for_essay.md` ánh xạ kết quả → mục tiểu luận → file nguồn, kèm mục **Công khai và sai lệch so với đăng ký trước**.

In [ ]:
show_table("s8_sha256_appendix.csv", 4)
print((TABLES / "summary_for_essay.md").read_text(encoding="utf-8")[:1500], "…")

## s9 – Kiểm định ngoài trên RCT thật (Hillstrom)

**Mục tiêu.** Kiểm định phương pháp (CATE/Qini, ước lượng giá trị chính sách, MILP) trên dữ liệu có ngẫu nhiên hóa thật, độc lập với Online Retail II.

**Phương pháp.** Chia 70/30 phân tầng; chọn learner bằng Qini CV; Qini và giá trị chính sách trên tập test với CI bootstrap; kết quả chính (conversion) khai báo trước, kết quả phụ (visit) tách riêng.
**Giới hạn.** Không phải kiểm định toàn pipeline. Hillstrom **không nêu giấy phép** nên không có trong repo; s9 tải bằng `sklift.datasets.fetch_hillstrom`.

**Liên quan:** mục IV.8, V · **CLO:** CLO3

In [ ]:
run("python src/s9_external_hillstrom.py")
show_table("s9_acceptance.csv", 8)
show_table("s9_ate_rct.csv", 6)
show_table("s9_qini_test.csv", 8)
show_fig("fig_s9_qini_curves.png")